# Track 1 — Predict Bike Demand

> **REFERENCE EXAMPLE — DO NOT SUBMIT UNCHANGED**
>
> This completed notebook demonstrates one reusable workflow. Run it to understand the decisions, then choose **a different dataset** for your final project and update the target or output, features, preprocessing, split, evaluation, and explanation to fit your data.


**Question:** How many bikes will people rent in a given hour?

This reference project uses the public **Bike Sharing in Washington, D.C.** dataset from Kaggle. You will build a chronological regression pipeline, compare it with a simple baseline, and inspect which decisions prevent misleading results.

### Finish line

- Download and inspect a real dataset.
- Explain why this problem needs a time-aware split.
- Remove columns that leak the answer.
- Compare a regression model with a median baseline using MAE and RMSE.
- Change one decision and explain what happened.

Dataset: https://www.kaggle.com/datasets/marklvl/bike-sharing-dataset


## 1. Setup

Kaggle's first-party kagglehub package downloads public datasets. If Kaggle asks for authentication in Colab, add your API token to Colab Secrets as KAGGLE_API_TOKEN; never paste a token into a shared notebook.


In [ ]:
%pip -q install "kagglehub>=1.0.3"


In [ ]:
from pathlib import Path
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
dataset_dir = Path(kagglehub.dataset_download("marklvl/bike-sharing-dataset"))
hour_path = next(dataset_dir.rglob("hour.csv"))
print("Using:", hour_path)


## 2. Inspect before modeling

One row represents one hour. The target is cnt, the total rentals in that hour.

Before choosing a model, check shape, types, missing values, and the target distribution. This is where we look for columns that would be unavailable when making a real prediction.


In [ ]:
df = pd.read_csv(hour_path)
display(df.head())
print("shape:", df.shape)
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame("missing"))
display(df["cnt"].describe().to_frame().T)


In [ ]:
df["timestamp"] = pd.to_datetime(df["dteday"]) + pd.to_timedelta(df["hr"], unit="h")
df = df.sort_values("timestamp").reset_index(drop=True)

df.plot(x="timestamp", y="cnt", figsize=(13, 4), alpha=0.65, title="Hourly bike demand over time")
plt.ylabel("rentals")
plt.show()


## 3. Make the key data decisions

**Why a chronological split?** A random split would let the model train on later dates and test on earlier ones. That does not match forecasting future demand.

**Why remove casual and registered?** They add up exactly to cnt. At prediction time we would not already know those counts, so using them would leak the answer.

**Why one-hot encode calendar codes?** Values such as season 4 are labels, not four times season 1.


In [ ]:
target = "cnt"
numeric_features = ["temp", "atemp", "hum", "windspeed"]
categorical_features = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]
features = numeric_features + categorical_features

split_index = int(len(df) * 0.80)
train = df.iloc[:split_index].copy()
test = df.iloc[split_index:].copy()

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

print("train:", train["timestamp"].min(), "to", train["timestamp"].max())
print("test: ", test["timestamp"].min(), "to", test["timestamp"].max())


## 4. Establish a baseline

The median baseline predicts the same typical demand for every hour. A more complex model is useful only if it improves on this simple starting point.


In [ ]:
def regression_report(name, truth, predictions):
    mae = mean_absolute_error(truth, predictions)
    rmse = mean_squared_error(truth, predictions) ** 0.5
    return {"model": name, "MAE": mae, "RMSE": rmse}

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
results = [regression_report("Median baseline", y_test, baseline_predictions)]
pd.DataFrame(results)


## 5. Build a reproducible pipeline

The pipeline learns category encodings and the model from training data only. Keeping these steps together prevents train/test inconsistencies and makes the workflow easier to adapt.

**Why do we leave the numeric columns unchanged?** A random forest makes threshold-based splits, so it does not require feature scaling. The categorical codes still need one-hot encoding because values such as season 4 are labels rather than quantities.


In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model = Pipeline(steps=[
    ("prepare", preprocess),
    ("model", RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

model.fit(X_train, y_train)
model_predictions = model.predict(X_test)
results.append(regression_report("Random forest pipeline", y_test, model_predictions))
pd.DataFrame(results).sort_values("MAE")


## 6. Inspect predictions and errors

A single metric can hide when the model struggles. Plot the first two weeks of the test period and then examine the largest errors.


In [ ]:
comparison = test[["timestamp", "cnt"]].copy()
comparison["prediction"] = model_predictions
comparison["absolute_error"] = (comparison["cnt"] - comparison["prediction"]).abs()

comparison.head(24 * 14).plot(
    x="timestamp", y=["cnt", "prediction"], figsize=(13, 4), title="First two test weeks"
)
plt.ylabel("rentals")
plt.show()

display(comparison.nlargest(10, "absolute_error"))


## 7. Try one change

Choose exactly one so you can attribute the result:

- remove weather features and compare the metrics;
- change the test size from 20% to the final 10%;
- compare a different regression model;
- predict daily rather than hourly demand;
- add a feature that represents rush hour.

Record the old metric, new metric, and one reason the change may have helped or hurt.


In [ ]:
# Example adaptation: add a rush-hour feature.
adapted_df = df.copy()
adapted_df["rush_hour"] = adapted_df["hr"].isin([7, 8, 9, 16, 17, 18]).astype(int)

# TODO: add "rush_hour" to the feature list, rebuild the same chronological split,
# fit the pipeline again, and compare MAE/RMSE with the reference result.


## Required next step: choose your own dataset

The dataset in this notebook is only the completed example. For your final project, find another time-based regression dataset such as energy use, traffic, sales, weather, or attendance.

Before changing code, write down:

1. the new dataset source and license;
2. what one row or example represents;
3. the new target or desired output;
4. possible leakage or unavailable information;
5. the split strategy and why it matches real use;
6. the preprocessing steps that must change;
7. the baseline and main metric; and
8. one visible result you will produce with your data.

You may reuse the overall workflow and model as a starting point. You may not use the unchanged reference notebook as the final project.


## Completion checklist

- [ ] I can explain one row, the target, and the intended user.
- [ ] I removed casual and registered and can explain the leakage.
- [ ] I used a chronological split.
- [ ] I compared the model with a median baseline.
- [ ] I inspected at least five large errors.
- [ ] I changed one decision and compared the result.
- [ ] I wrote one limitation of this dataset or evaluation.
